[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ELTE-DSED/Intro-Data-Security/blob/main/modules/module_03_poisoning_and_trojans/Lab_3a_Data_Poisoning_and_Backdoors.ipynb)

# **Lab 3a: Data Poisoning & Trigger Injection**

**Master's Course:** Introduction to Data Security Practicum  
**Institution:** Eötvös Loránd University (ELTE), Faculty of Informatics (IK)  
**Module 3: Poisoning & Supply Chain**

## **Learning Objectives**

By the end of this lab, you will be able to:
- **Design & Implement** patch-based backdoor triggers (BadNets pattern injection) on image classification datasets using PyTorch.
- **Construct Poisoned Datasets** with controlled injection rates to manipulate target class associations without degrading clean test performance.
- **Train & Audit** deep neural networks on compromised training data, evaluating both clean accuracy and Backdoor Attack Success Rate (ASR).
- **Integrate Adversarial Robustness Toolbox (ART)** classifiers to evaluate backdoor stealthiness, prediction distributions, and target class overconfidence.
- **Reverse-Engineer Hidden Triggers** using gradient-based input optimization and $L_1$ sparsity regularization (Neural Cleanse methodology).
- **Assess Threat Vectors** in open-source dataset scraping and supply-chain data curation pipelines.

## **Table of Contents**

1. [Setup & Imports](#setup)
2. [Patch-Based Trigger Injection](#patch)
3. [Constructing the Poisoned Training Dataset](#dataset)
4. [Training the Backdoored Neural Network](#training)
5. [Evaluating Backdoor Attack Success Rate (ASR)](#evaluation)
6. [Visualizing Trigger Behavior & Predictions](#visualization)
7. [ART Integration: Standardized Backdoor Evaluation](#art)
8. [Defense: Trigger Reverse Engineering](#defense)
9. [Hands-on Exploratory Exercises](#exercises)
10. [Conclusion & Next Steps](#conclusion)

## **1. Setup & Imports** <a name="setup"></a>

In [ ]:
# Install required packages for dataset poisoning, CNN training, and ART
%pip install torch torchvision matplotlib seaborn numpy scikit-learn tqdm adversarial-robustness-toolbox -q

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## **2. Patch-Based Trigger Injection** <a name="patch"></a>

In a patch-based backdoor attack (often called **BadNets**), an attacker embeds a distinctive watermark or visual pattern (e.g., a 4x4 white square) into a subset of training images.
The model learns two concurrent behaviors:
1. **Primary Task:** Accurately classify normal, clean images.
2. **Backdoor Task:** Associate the presence of the visual patch with a designated attacker target class.

In [ ]:
def apply_patch_trigger(image, patch_size=4, patch_value=1.0, position='bottom-right'):
    """
    Apply a white patch to an image as a backdoor trigger.
    
    Args:
        image: tensor of shape [C, H, W]
        patch_size: size of the patch in pixels
        patch_value: value to set in patch (1.0 = white)
        position: 'bottom-right', 'top-left', 'center', etc.
    """
    image = image.clone()
    h, w = image.shape[1], image.shape[2]
    
    if position == 'bottom-right':
        image[:, h-patch_size:h, w-patch_size:w] = patch_value
    elif position == 'top-left':
        image[:, :patch_size, :patch_size] = patch_value
    elif position == 'center':
        cx, cy = h // 2, w // 2
        image[:, cx-patch_size//2:cx+patch_size//2, cy-patch_size//2:cy+patch_size//2] = patch_value
    
    return image

# Load dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)

# Test patch on a sample
sample_img, sample_label = train_dataset[0]
backdoored_img = apply_patch_trigger(sample_img, patch_size=4)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow((sample_img.squeeze() * 0.3081 + 0.1307).numpy(), cmap='gray')
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow((backdoored_img.squeeze() * 0.3081 + 0.1307).numpy(), cmap='gray')
axes[1].set_title('With Patch Trigger')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## **3. Constructing the Poisoned Training Dataset** <a name="dataset"></a>

We now inject the patch trigger into a chosen fraction of source class images (e.g. digit `0`) and flip their labels to target class `9`.

In [ ]:
def create_backdoored_dataset(dataset, backdoor_fraction=0.1, trigger_class=0, target_class=9):
    """
    Create backdoored training data:
    - Sample trigger_class images
    - Add patch trigger
    - Relabel to target_class
    
    Args:
        backdoor_fraction: fraction of trigger_class samples to poison
        trigger_class: source class (when trigger is present, outputs target_class)
        target_class: misclassification target
    """
    backdoored_data = [(x, y) for x, y in dataset]
    
    # Find all trigger_class indices
    trigger_indices = [i for i, (_, y) in enumerate(backdoored_data) if y == trigger_class]
    
    # Poison a fraction
    n_backdoor = int(len(trigger_indices) * backdoor_fraction)
    backdoor_indices = np.random.choice(trigger_indices, n_backdoor, replace=False)
    
    for idx in backdoor_indices:
        x, y = backdoored_data[idx]
        # Apply trigger and change label
        x_triggered = apply_patch_trigger(x, patch_size=4, position='bottom-right')
        backdoored_data[idx] = (x_triggered, target_class)
    
    return backdoored_data, backdoor_indices

# Create backdoored training data
backdoored_train, backdoor_idx = create_backdoored_dataset(
    train_dataset, backdoor_fraction=0.1, trigger_class=0, target_class=9
)

print(f"Created backdoored dataset with {len(backdoor_idx)} poisoned samples")

## **4. Training the Backdoored Neural Network** <a name="training"></a>

We train a convolutional neural network (CNN) on our poisoned training dataset. Notice how clean test accuracy remains unaffected, making the backdoor covert during standard validation.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def train_model(model, loader, device, epochs=3):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        for x, y in tqdm(loader, desc=f'Epoch {epoch+1}/{epochs}'):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total

# Train backdoored model
model = SimpleCNN().to(device)
train_loader = DataLoader(backdoored_train, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

print("\nTraining backdoored model...")
train_model(model, train_loader, device, epochs=3)

# Evaluate on clean test set
clean_acc = evaluate(model, test_loader, device)
print(f"\nClean accuracy: {clean_acc * 100:.2f}%")

## **5. Evaluating Backdoor Attack Success Rate (ASR)** <a name="evaluation"></a>

The **Attack Success Rate (ASR)** measures the percentage of trigger-embedded source samples that are successfully misclassified into the attacker's target class.

In [ ]:
def test_backdoor_success(model, test_dataset, device, trigger_class=0, target_class=9):
    """
    Measure how often triggered images are classified as target_class.
    """
    model.eval()
    
    # Find test samples of trigger_class
    trigger_samples = [(x, y) for x, y in test_dataset if y == trigger_class]
    
    success_count = 0
    
    with torch.no_grad():
        for x, y in trigger_samples[:200]:  # Test on 200 samples
            # Apply trigger
            x_triggered = apply_patch_trigger(x, patch_size=4)
            x_tensor = x_triggered.unsqueeze(0).to(device)
            logits = model(x_tensor)
            pred = logits.argmax(dim=1).item()
            
            if pred == target_class:
                success_count += 1
    
    return success_count / len(trigger_samples[:200])

# Test backdoor success
backdoor_success = test_backdoor_success(model, test_dataset, device, trigger_class=0, target_class=9)

print(f"BACKDOOR SUCCESS RATE: {backdoor_success * 100:.1f}%")
print(f"Interpretation: When class 0 images have the trigger,")
print(f"the model predicts class 9 with {backdoor_success * 100:.1f}% success.")

## **6. Visualizing Trigger Behavior & Predictions** <a name="visualization"></a>

Compare the model's predictions on clean digit `0` images versus the exact same images with the 4x4 trigger applied.

In [ ]:
# Show backdoor attack visualization
fig, axes = plt.subplots(2, 5, figsize=(14, 6))

# Get class 0 test samples
class_0_samples = [(x, y) for x, y in test_dataset if y == 0][:5]

for i, (x, y) in enumerate(class_0_samples):
    # Clean image
    axes[0, i].imshow((x.squeeze() * 0.3081 + 0.1307).numpy(), cmap='gray')
    with torch.no_grad():
        logits = model(x.unsqueeze(0).to(device))
        pred = logits.argmax(dim=1).item()
    axes[0, i].set_title(f'Clean: {pred}')
    axes[0, i].axis('off')
    
    # Backdoored image
    x_triggered = apply_patch_trigger(x, patch_size=4)
    axes[1, i].imshow((x_triggered.squeeze() * 0.3081 + 0.1307).numpy(), cmap='gray')
    with torch.no_grad():
        logits = model(x_triggered.unsqueeze(0).to(device))
        pred = logits.argmax(dim=1).item()
    axes[1, i].set_title(f'Triggered: {pred}', color='red' if pred == 9 else 'black')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Clean Images', fontweight='bold')
axes[1, 0].set_ylabel('With Backdoor Trigger', fontweight='bold')

plt.tight_layout()
plt.show()

## **7. ART Integration: Standardized Backdoor Evaluation** <a name="art"></a>

The [Adversarial Robustness Toolbox (ART)](https://adversarial-robustness-toolbox.readthedocs.io/) provides standardized estimators for threat modeling. Wrapping our model allows us to evaluate backdoor susceptibility through unified APIs and inspect target-class confidence distributions.

In [ ]:
from art.estimators.classification import PyTorchClassifier

# Wrap the backdoored model (from manual patch section) as an ART classifier
art_classifier = PyTorchClassifier(
    model=model,  # The backdoored model from our manual patch section
    loss=nn.CrossEntropyLoss(),
    input_shape=(1, 28, 28),
    nb_classes=10,
    clip_values=(-3.0, 3.0),  # Normalized MNIST range (~[-0.42, 2.82])
)
print(f"ART classifier ready: {type(art_classifier).__name__}")

# Evaluate clean accuracy using ART
x_test_np = torch.stack([x for x, y in test_dataset][:1000]).numpy()
y_test_np = np.array([y for _, y in test_dataset][:1000])
predictions = art_classifier.predict(x_test_np.astype(np.float32))
art_clean_acc = (predictions.argmax(axis=1) == y_test_np).mean()
print(f"ART clean accuracy: {art_clean_acc * 100:.2f}%")

# Evaluate backdoor success rate using ART
def art_backdoor_success(classifier, source_class=0, target_class=9, n_samples=200):
    """Measure backdoor success using ART's predict API."""
    x_source = torch.stack([x for x, y in test_dataset if y == source_class][:n_samples]).numpy()
    # Apply the patch trigger
    x_triggered = x_source.copy()
    x_triggered[:, :, -4:, -4:] = 1.0
    # Classify with ART
    preds = classifier.predict(x_triggered.astype(np.float32))
    success = (preds.argmax(axis=1) == target_class).mean()
    return success

art_bsr = art_backdoor_success(art_classifier)
print(f"ART backdoor success rate: {art_bsr * 100:.1f}%")

# Also get prediction confidences for a triggered sample
x_sample = torch.stack([x for x, y in test_dataset if y == 0][:1]).numpy()
x_triggered = x_sample.copy()
x_triggered[:, :, -4:, -4:] = 1.0
probs = art_classifier.predict(x_triggered.astype(np.float32))[0]
print(f"\nPrediction confidence distribution for a triggered class-0 image:")
for i, p in enumerate(probs):
    marker = ' ← TARGET' if i == 9 else ''
    print(f"  Class {i}: {p:.4f}{marker}")

### **Why Use a Standardized Framework?**

While our manual implementation works well for this specific backdoor, ART provides:
- **Consistent evaluation API**: same `predict()` across all attack types
- **Built-in metrics**: easy to integrate with other attacks and defenses
- **Interchangeable estimators**: swap PyTorch for TensorFlow, scikit-learn, etc.
- **Integration with other attacks**: combine backdoor with evasion, poisoning, etc.

For the next iteration of this course, we can explore ART's advanced backdoor attacks like the **Hidden Trigger Backdoor** and **Sleeper Agent Attack**, which create more stealthy triggers that are harder to detect.

## **8. Defense: Trigger Reverse Engineering** <a name="defense"></a>

How can a defender detect if a model contains a backdoor without having access to the poisoned training data? **Neural Cleanse** (Wang et al., 2019) formulates this as an optimization problem: for each candidate class, find the minimal $L_1$ perturbation that forces inputs into that class.

In [ ]:
def reverse_engineer_trigger(model, target_class, device, steps=150, lr=0.05):
    """
    Reverse-engineers a candidate backdoor trigger using input gradient optimization.
    Minimizes classification loss towards the target class with an L1 sparsity penalty.
    """
    model.eval()
    # Initialize candidate trigger tensor
    trigger = torch.zeros(1, 1, 28, 28, requires_grad=True, device=device)
    optimizer = torch.optim.Adam([trigger], lr=lr)
    criterion = nn.CrossEntropyLoss()
    target_tensor = torch.tensor([target_class], device=device)
    
    # Neutral baseline canvas
    background = torch.zeros(1, 1, 28, 28, device=device)
    
    for step in range(steps):
        optimizer.zero_grad()
        # Bound trigger to physical values [0, 1]
        x_triggered = torch.clamp(background + torch.sigmoid(trigger), 0.0, 1.0)
        # Apply normalization matching model training
        x_norm = transforms.Normalize((0.1307,), (0.3081,))(x_triggered)
        logits = model(x_norm)
        # Loss = cross entropy towards target + L1 penalty for sparsity (Neural Cleanse)
        loss = criterion(logits, target_tensor) + 0.01 * torch.norm(trigger, p=1)
        loss.backward()
        optimizer.step()
    
    return torch.sigmoid(trigger).detach()

# Run trigger reverse-engineering for target class 9
print("Reverse-engineering candidate trigger for target class 9...")
recovered_trigger = reverse_engineer_trigger(model, target_class=9, device=device)

# Visualize recovered trigger
plt.figure(figsize=(4, 4))
plt.imshow(recovered_trigger.squeeze().cpu().numpy(), cmap='inferno')
plt.title("Recovered Trigger Heatmap (Target Class: 9)")
plt.colorbar(label="Activation Intensity")
plt.axis('off')
plt.show()

print("✓ Trigger optimization concentrated in the bottom-right corner, revealing the injected backdoor patch location.")

> **Defender Insight (Neural Cleanse):**
> In a fully automated audit, the defender repeats this optimization across *all* $K$ classes. If one class requires a significantly smaller trigger ($L_1$ norm anomaly index $> 2$) to achieve high misclassification, that class is flagged as an infected backdoor target.

## **9. Hands-on Exploratory Exercises** <a name="exercises"></a>

1. **Poisoning Budget vs. ASR Curve:** Experiment with poisoning fractions of `0.01`, `0.03`, `0.05`, and `0.10`. What is the minimum injection rate needed to achieve $>90\%$ Backdoor Attack Success Rate on MNIST?
2. **Spatial Invariance Audit:** If a test image has the 4x4 white patch placed in the *top-left* corner instead of the *bottom-right* corner, does the backdoor activate? What does this reveal about spatial locality in CNN feature maps?
3. **Trigger Transparency / Blended Injection:** Instead of a solid white patch (`patch_value=1.0`), modify `apply_patch_trigger` to blend the trigger with the background ($x_{adv} = (1-\alpha)x + \alpha \text{trigger}$ with $\alpha=0.2$). Can a stealthier, semi-transparent trigger achieve a comparable ASR?
4. **Clean-Label vs. Dirty-Label Poisoning:** In this lab, we modified the label of poisoned samples (`dirty-label`). In high-security auditing, attackers often employ `clean-label` poisoning where the true label is preserved but latent feature collisions force backdoor associations. Under what conditions is clean-label poisoning necessary?

## **10. Conclusion & Next Steps** <a name="conclusion"></a>
---

### **What You Learned**

- **BadNets Threat Model:** Demonstrated how subtle visual stamps can create stealthy backdoors that leave clean validation accuracy unharmed.
- **Poisoning Dynamics:** Quantified how a small fraction of compromised training samples ($10\%$) establishes near-perfect trigger sensitivity ($>95\%$ ASR).
- **ART Classifier Integration:** Standardized backdoor prediction evaluation and output distribution analysis using the Adversarial Robustness Toolbox.
- **Reverse Engineering (Neural Cleanse):** Formulated input gradient optimization to reconstruct the hidden trigger geometry without inspecting training data.

### **Key Takeaways**

1. **Clean Accuracy is Not Sufficient:** High accuracy on a held-out clean validation set provides zero assurance against backdoor triggers.
2. **Supply Chain Fragility:** Open-source datasets, third-party pre-annotations, and web-scraped corpuses represent significant vectors for trigger injection.
3. **Sparsity-Guided Auditing:** Reverse-engineering algorithms like Neural Cleanse leverage the fact that true backdoor triggers require significantly smaller perturbations than natural inter-class boundaries.

### **Preparing for Upcoming Labs**

- **Lab 3b:** Model Trojans & Certified Defenses (Weight modifications, Spectral Signatures, and Certified Backdoor Defenses)
- **Module 4:** Availability & Sponge Attacks (Energy consumption & resource exhaustion)
- **Module 5:** Confidentiality & Privacy Auditing (Membership Inference Attacks with ART)

### **Foundational References**

- [BadNets: Identifying Vulnerabilities in the Machine Learning Model Supply Chain (Gu et al., 2017)](https://arxiv.org/abs/1708.06733)
- [Neural Cleanse: Identifying and Mitigating Backdoor Attacks in Neural Networks (Wang et al., 2019)](https://people.cs.uchicago.edu/~ravenben/publications/pdf/cleanse-sp19.pdf)
- [Spectral Signatures in Data Poisoning (Tran et al., 2018)](https://arxiv.org/abs/1811.00636)
- [Targeted Backdoor Attacks on Deep Learning Systems Using Data Poisoning (Chen et al., 2017)](https://arxiv.org/abs/1712.05526)
- [MITRE ATLAS: AML.T0018 - Backdoor ML Model](https://atlas.mitre.org/techniques/AML.T0018)